In [ ]:
import torch
from descope.inference import InferenceForATAC

### 1. Initialize Inference Tool

In [ ]:
infer = InferenceForATAC(
    test_csv_template_fp="/fse/home/wupengpeng/perturbation_data_origin/ATAC/GM12878/Test_Info_GM12878.csv",
    adata="/fse/home/wupengpeng/perturbation_data_origin/ATAC/GM12878/GM12878.h5ad",
    pretrained_model_name_or_path="./trainer_output/GM12878/last_model",
    pert_col="perturbation",
    ctrl_name="control"
)

### 2. Start Inference Process

- **Note:**
    
    - **!!! Always use real control cells, i.e., set `use_generated_control_cells=False` !!!**

    - **!!! Never set `use_generate_control_cells=True` !!!**

In [ ]:
infer.inference(
    gene_embs_file="/fse/home/wupengpeng/DeSCOPE/ESM2_pert_features.pt",
    device=torch.device("cuda:0"),
    batch_size=256,
    use_generated_control_cells=False
)

### 3. Calculate Metrics

#### 3.1 Compute the built-in metrics of cell-eval

In [ ]:
# Select metrics you want to calculate
infer.all_metrics

In [ ]:
metrics_to_calculate = [
    'pearson_delta',
    'overlap_at_N',
    'overlap_at_50',
    'overlap_at_100',
    'overlap_at_200',
    'overlap_at_500',
    'discrimination_score_l1',
    'mae'
]

results, agg_results, evaluator = infer.compute_metrics(
    adata_pred=infer.adata_pred,
    adata_real=infer.adata_test_ground_truth,
    metrics_to_calculate=metrics_to_calculate,
    de_pred=None,
    de_real=None,
    control_pert=infer.ctrl_name,
    pert_col=infer.pert_col,
    de_method="wilcoxon",
    num_threads=32,
    outdir="./cell-eval-outdir/GM12878"
)

In [ ]:
agg_results

#### 3.2 Compute Extra Metrics

In [ ]:
# pearson
pearson, pearson_mean = infer.extra_metrics_func.pearson(evaluator.anndata_pair)
pearson_mean

In [ ]:
# person_delta_on_topk_de
pearson_delta_on_topk_de, pearson_delta_on_topk_de_mean = infer.extra_metrics_func.pearson_delta_on_topk_de(
    data=evaluator.anndata_pair,
    de_real="./cell-eval-outdir/GM12878/real_de.csv",
    topk=20
)
pearson_delta_on_topk_de_mean

In [ ]:
# direction_match_on_topk_de
up, down, up_mean, down_mean = infer.extra_metrics_func.direction_match_on_topk_de(
    data=evaluator.anndata_pair,
    de_real="./cell-eval-outdir/GM12878/real_de.csv",
    topk=100,
    separate_up_down_regulated=True
)
up_mean, down_mean

### 4. Aggregate Results

In [ ]:
results = results.to_pandas()
results["pearson"] = results["perturbation"].map(pearson)
results["pearson_delta_on_topk_de"] = results["perturbation"].map(pearson_delta_on_topk_de)
results["direction_match_on_topk_up_de"] = results["perturbation"].map(up)
results["direction_match_on_topk_down_de"] = results["perturbation"].map(down)
results.to_csv("./cell-eval-outdir/GM12878/results_for_each_perturbation.csv")

### 5. Save Predicted AnnData

In [ ]:
infer.write_h5ad(
    adata=infer.adata_pred,
    save_path="./cell-eval-outdir/GM12878/descope_preds.h5ad"
)